# 04 · Indicadores de riesgo (red flags)

**Objetivo:** analizar las alertas que calcula el pipeline y priorizar procesos para revisión.

> **Importante:** una alerta **no** prueba una irregularidad ni atribuye responsabilidad a ninguna
> entidad o proveedor. Es una señal estadística para priorizar la revisión humana.
> Metodología en `docs/08_indicadores_de_riesgo.md`.

In [ ]:
# Configuración del entorno (ejecutar primero)
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd

# Raíz del proyecto: la carpeta actual o la primera superior con pyproject.toml.
CWD = Path.cwd().resolve()
ROOT = next((p for p in (CWD, *CWD.parents) if (p / "pyproject.toml").is_file()), CWD)
sys.path.insert(0, str(ROOT / "src"))  # permite importar el paquete aunque no esté instalado

CURATED = ROOT / "data" / "curated"      # capa gold
PROCESSED = ROOT / "data" / "processed"  # capa silver
if not (CURATED / "fact_proceso.parquet").exists():
    raise FileNotFoundError(
        "No hay datos procesados. Ejecuta primero en la terminal: contrataciones run --source sample"
    )

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)
PALETA = ["#1F2A44", "#3E6A8A", "#7FA7C2", "#C9A227", "#B5533C", "#5B8C5A", "#8C8C8C"]
plt.rcParams.update({
    "figure.figsize": (10, 4.5), "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "axes.prop_cycle": plt.cycler(color=PALETA),
    "axes.titleweight": "bold", "axes.titlesize": 12,
})
millones = mticker.FuncFormatter(lambda x, _: f"{x / 1e6:,.0f} M")
print("Proyecto:", ROOT)

In [ ]:
def cargar(tabla):
    return pd.read_parquet(CURATED / f"{tabla}.parquet")

fp, alertas, indicadores = cargar("fact_proceso"), cargar("fact_alerta"), cargar("dim_indicador")
entidades, metodos, agg = cargar("dim_entidad"), cargar("dim_metodo"), cargar("agg_entidad_anio")
indicadores[["codigo", "nombre", "peso", "referencia"]]

## 1. ¿Cuántos procesos marca cada indicador?

In [ ]:
conteo = (alertas.groupby("codigo")["ocid"].nunique()
          .reindex(indicadores["codigo"]).fillna(0).astype(int))
etiquetas = indicadores.set_index("codigo")["nombre"]
ax = conteo.rename(index=lambda c: f"{c} · {etiquetas[c]}").sort_values().plot.barh(
    title="Procesos marcados por indicador", color=PALETA[0])
ax.set_ylabel(""); plt.show()
(conteo / len(fp)).map("{:.1%}".format)

## 2. Distribución del nivel de riesgo

In [ ]:
orden = ["Sin alertas", "Bajo", "Medio", "Alto"]
niveles = fp["nivel_riesgo"].value_counts().reindex(orden).fillna(0)
colores = ["#C8CFD8", "#7FA7C2", "#C9A227", "#B5533C"]
plt.bar(niveles.index, niveles.values, color=colores)
plt.title("Procesos por nivel de riesgo"); plt.ylabel("N.° de procesos"); plt.show()
niveles

## 3. Competencia: tasa de postor único por procedimiento

In [ ]:
base = fp.merge(metodos, on="metodo_key")
competitivos = base[base["es_competitivo"] & ~base["estado"].isin(["cancelled", "withdrawn"])]
tasa = (competitivos.assign(unico=competitivos["n_postores"].eq(1))
        .groupby("metodo_detalle")["unico"].mean().sort_values())
ax = tasa.plot.barh(title="% de procesos con un solo postor (procedimientos competitivos)",
                    color=PALETA[1])
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0)); ax.set_ylabel(""); plt.show()

## 4. Concentración de proveedores por entidad (HHI)

El índice Herfindahl-Hirschman va de 0 a 10 000. Como referencia: < 1 500 baja concentración,
1 500-2 500 moderada y > 2 500 alta.

In [ ]:
hhi = (agg[agg["n_procesos"] >= 5].merge(entidades[["entidad_key", "nombre"]], on="entidad_key")
       .sort_values("hhi", ascending=False).head(10))
hhi[["nombre", "anio", "n_procesos", "n_proveedores", "monto_adjudicado_pen", "hhi"]]

## 5. Procesos priorizados para revisión

In [ ]:
detalle = alertas.groupby("ocid").agg(alertas=("codigo", lambda s: ", ".join(sorted(s))))
priorizados = (fp.merge(detalle, on="ocid").merge(entidades[["entidad_key", "nombre"]], on="entidad_key")
               .sort_values(["puntaje_riesgo", "valor_referencial_pen"], ascending=False)
               .head(15))
priorizados[["ocid", "nomenclatura", "nombre", "valor_referencial_pen", "n_postores",
             "puntaje_riesgo", "nivel_riesgo", "alertas"]]

## Conclusiones

* Revisa primero los procesos con puntaje alto **y** monto alto (mayor impacto potencial).
* Calibra los umbrales en `config/risk_rules.yaml` (por ejemplo, plazos mínimos por procedimiento)
  para reducir falsos positivos y vuelve a ejecutar el pipeline.
* Con datos reales del OECE, revisa en el reporte de calidad la regla *plazo de presentación no
  informado*: si afecta a la mayoría de procesos, evalúa RF02 con `period: consultas`.
* Estos resultados alimentan la página **Alertas de riesgo** del dashboard de Power BI.